# Глава 8. Метрики качества классификации

Код к главе 8 книги 1: матрица ошибок, точность и полнота, выбор порога, ROC-кривая и AUC,
кривая точность–полнота, логарифмическая потеря и метрики scikit-learn (листинг 8.1).

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markpolyak/ml-textbook-code/blob/main/book1/ch08_metrics.ipynb)

In [ ]:
# В Colab ставим пакет mlbook с данными книги (локально: pip install -e . в корне репозитория)
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/markpolyak/ml-textbook-code

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mlbook import examples, modules

## Сквозной пример: десять модулей

Табл. 8.2: вероятности ошибки $p$, которые выдала модель, и истинные ответы.

In [ ]:
sc = examples.scores10()
p, y = sc["p"].to_numpy(), sc["y"].to_numpy()
sc

## Матрица ошибок, точность и полнота

Порог 0,5 и порог 0,35. Порядок строк и столбцов задан как в табл. 8.1: сначала класс $+1$.

In [ ]:
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score)

for thr in (0.5, 0.35):
    y_hat = np.where(p >= thr, 1, -1)
    (tp, fn), (fp, tn) = confusion_matrix(y, y_hat, labels=[1, -1])
    print(f"порог {thr}: TP = {tp}, FN = {fn}, FP = {fp}, TN = {tn}")
    print(f"  accuracy = {accuracy_score(y, y_hat):.2f}, precision = {precision_score(y, y_hat):.2f},",
          f"recall = {recall_score(y, y_hat):.2f}, F1 = {f1_score(y, y_hat):.2f},",
          f"специфичность = {tn / (tn + fp):.2f}")
print("классификатор «всегда −1»: accuracy =", accuracy_score(y, -np.ones_like(y)))

**F-мера** близка к меньшему из чисел: при точности 1 и полноте 0,1 среднее арифметическое
0,55, а $F_1 \approx 0{,}18$. $F_2$ важнее полнота, $F_{0,5}$ — точность.

In [ ]:
P, R = 1.0, 0.1
f_beta = lambda b: (1 + b ** 2) * P * R / (b ** 2 * P + R)
print(f"F1 = {f_beta(1):.2f}, F2 = {f_beta(2):.2f}, F0.5 = {f_beta(0.5):.2f}")

**Несколько классов.** Макроусреднение — среднее метрик по классам, микроусреднение —
метрика по суммарным TP, FP, FN. Плохое качество на редком классе заметно только в первом.

In [ ]:
y3 = np.array([0] * 8 + [1] * 8 + [2] * 2)
y3_hat = np.array([0] * 8 + [1] * 8 + [0, 1])          # редкий класс 2 не распознан ни разу
for avg in ("macro", "micro"):
    print(avg, round(recall_score(y3, y3_hat, average=avg), 3))

## Порог и цена ошибок

Если пропуск стоит $c_+$, а ложная тревога $c_-$, выгодный порог $c_-/(c_- + c_+)$.

In [ ]:
for c_minus, c_plus in [(1, 1), (1, 4), (1, 9), (1, 20)]:
    print(f"c− = {c_minus}, c+ = {c_plus}: порог {c_minus / (c_minus + c_plus):.3f}")

## ROC-кривая и AUC

AUC — доля верно упорядоченных пар «положительный — отрицательный».

In [ ]:
from sklearn.metrics import roc_auc_score

pos, neg = p[y == 1], p[y == -1]
pairs = pos[:, None] > neg[None, :]
print(f"верно упорядочено {pairs.sum()} пар из {pairs.size}: {pairs.mean():.3f}")
print("roc_auc_score:", roc_auc_score(y, p))
print("после возведения оценок в квадрат:", roc_auc_score(y, p ** 2))

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_curve

fpr, tpr, thr_roc = roc_curve(y, p)
prec, rec, _ = precision_recall_curve(y, p)
fig, (a, b) = plt.subplots(1, 2, figsize=(9, 4.2))
a.step(fpr, tpr, where="post")
a.plot([0, 1], [0, 1], "--", color="0.6")
a.plot([1 / 6, 1 / 3], [0.75, 1], "o", color="C1")
a.set_xlabel("FPR")
a.set_ylabel("TPR (полнота)")
a.set_title(f"а) ROC, AUC = {roc_auc_score(y, p):.3f}")
b.plot(rec, prec, "o-")
b.axhline(np.mean(y == 1), linestyle="--", color="0.6")
b.set_xlabel("полнота")
b.set_ylabel("точность")
b.set_title(f"б) точность–полнота, AP = {average_precision_score(y, p):.2f}")
for ax in (a, b):
    ax.set_aspect("equal")
    ax.grid(True)
plt.show()

## Логарифмическая потеря

Log-loss чувствительна к величине вероятностей. Наибольший вклад даёт модуль 3: модель
уверенно сочла его дефектным, а ошибок в нём нет.

In [ ]:
from sklearn.metrics import log_loss

yt = (y + 1) // 2                                    # метки 0 и 1
loss_i = -(yt * np.log(p) + (1 - yt) * np.log(1 - p))
print("log-loss:", round(log_loss(yt, p), 2), " вклады:", loss_i.round(2))

## Метрики модели на данных о модулях

Обучим логистическую регрессию, как в гл. 7, и применим листинг 8.1 со сниженным
порогом 0,35.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = modules.load()
X_train, X_test, y_train, y_test = train_test_split(
    df[modules.FEATURES], df["ошибка"], test_size=0.3, stratify=df["ошибка"], random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import log_loss, roc_auc_score

p = model.predict_proba(X_test)[:, 1]  # P(класс 1)
y_pred = (p >= 0.35).astype(int)       # свой порог
print(confusion_matrix(y_test, y_pred))
# точность, полнота и F1 по каждому классу
print(classification_report(y_test, y_pred))
print(roc_auc_score(y_test, p), log_loss(y_test, p))

Сравнение порогов и базовые модели: правило «всегда 0» и правило по одному признаку
(«модуль подозрителен, если в нём больше тысячи строк»).

In [ ]:
def report(name, y_hat):
    print(f"{name:26} accuracy {accuracy_score(y_test, y_hat):.2f}  precision",
          f"{precision_score(y_test, y_hat, zero_division=0):.2f}  recall",
          f"{recall_score(y_test, y_hat):.2f}  F1 {f1_score(y_test, y_hat):.2f}")


for thr in (0.5, 0.35, 0.2):
    report(f"модель, порог {thr}", (p >= thr).astype(int))
report("всегда 0", np.zeros_like(y_test))
report("строк > 1000", (X_test["строк"] > 1000).astype(int))

Доля правильных ответов у правила «всегда 0» не хуже, чем у модели, хотя оно не находит
ни одного дефектного модуля. Снижение порога увеличивает полноту ценой точности; правило
по одному признаку уступает модели по всем метрикам.

## Задачи

**1.** TP = 40, FP = 60, FN = 10, TN = 890.

In [ ]:
tp, fp, fn, tn = 40, 60, 10, 890
P, R = tp / (tp + fp), tp / (tp + fn)
print(f"accuracy {(tp + tn) / 1000:.2f}, precision {P:.2f}, recall {R:.2f},",
      f"F1 {2 * P * R / (P + R):.2f}, специфичность {tn / (tn + fp):.2f}")

**2.** Табл. 8.2, порог 0,65.

In [ ]:
y_hat = np.where(sc["p"] >= 0.65, 1, -1)
print(precision_score(sc["y"], y_hat), recall_score(sc["y"], y_hat), round(f1_score(sc["y"], y_hat), 2))

**3.** AUC подсчётом пар: оценки 0,9; 0,8; 0,3 у положительных и 0,7; 0,2 у отрицательных.

In [ ]:
print(roc_auc_score([1, 1, 1, 0, 0], [0.9, 0.8, 0.3, 0.7, 0.2]))